# Unsloth Core — Dataset Generation

Generate NPC training data in Colab using template, Ollama, OpenAI, or Anthropic.


In [ ]:
# @title ⚙️ Dataset Generation Configuration
# @markdown Configure the NPC and technique before running generation below.

# --- NPC Identity ---
NPC_KEY = "history_guide"  # @param ["history_guide", "chef_assistant", "fitness_coach", "astronomy_guide"]
# @markdown The NPC key in snake_case (e.g. `history_guide`, `chef_assistant`).

# --- Dataset Technique ---
TECHNIQUE = "template"  # @param ["template", "ollama", "openai", "anthropic"]
# @markdown - **template**: Fast deterministic generation (no external deps)
# @markdown - **ollama**: Local LLM generation via Ollama (not available in Colab by default)
# @markdown - **openai**: OpenAI API generation (requires `OPENAI_API_KEY` secret)
# @markdown - **anthropic**: Anthropic API generation (requires `ANTHROPIC_API_KEY` secret)

# --- Seed ---
TEMPLATE_SEED = 42  # @param {type:"integer"}
# @markdown Seed for deterministic template generation.

# --- Ollama ---
OLLAMA_MODEL = "qwen2.5:7b"  # @param {type:"string"}
# @markdown Model name for the ollama technique.

# --- OpenAI ---
OPENAI_MODEL = "gpt-4o-mini"  # @param {type:"string"}
# @markdown Model name for the openai technique.

# --- Anthropic ---
ANTHROPIC_MODEL = "claude-sonnet-4-20250514"  # @param {type:"string"}
# @markdown Model name for the anthropic technique.

print("Configuration loaded:")
print(f"  NPC_KEY          = {NPC_KEY}")
print(f"  TECHNIQUE        = {TECHNIQUE}")
print(f"  TEMPLATE_SEED    = {TEMPLATE_SEED}")
print(f"  OLLAMA_MODEL     = {OLLAMA_MODEL}")
print(f"  OPENAI_MODEL     = {OPENAI_MODEL}")
print(f"  ANTHROPIC_MODEL  = {ANTHROPIC_MODEL}")


In [ ]:
# @title 📦 Setup: Clone Repository & Install Dependencies
# @markdown Mounts Google Drive, clones the repo, installs minimal deps.

import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/athargamedev/Unsloth_Core.git"
DRIVE_REPO_DIR = "/content/drive/MyDrive/Unsloth_Core"
FALLBACK_REPO_DIR = "/content/Unsloth_Core"

# --- Detect Runtime ---
is_colab = False
try:
    import google.colab  # type: ignore
    is_colab = True
except ImportError:
    pass

if is_colab:
    print("Running in Google Colab.")
    repo_dir = DRIVE_REPO_DIR
    use_persistent = True
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        if os.path.exists("/content/drive/MyDrive"):
            print("Google Drive mounted. Using persistent storage:", repo_dir)
        else:
            raise OSError("Google Drive mounted but MyDrive is not accessible.")
    except Exception as e:
        repo_dir = FALLBACK_REPO_DIR
        use_persistent = False
        print(f"Drive mount failed: {e}")
        print("Using ephemeral storage:", repo_dir)

    # Clone or pull the repository
    try:
        if not os.path.exists(repo_dir):
            os.makedirs(os.path.dirname(repo_dir), exist_ok=True)
            subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
            print("Cloned repository.")
        else:
            git_dir = os.path.join(repo_dir, ".git")
            if os.path.exists(git_dir) and os.path.isdir(git_dir):
                orig = os.getcwd()
                os.chdir(repo_dir)
                subprocess.run(["git", "pull"], check=False)
                os.chdir(orig)
                print("Pulled latest changes.")
            else:
                import shutil
                shutil.rmtree(repo_dir)
                subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
                print("Re-cloned repository (existing dir was not a git repo).")
    except OSError as e:
        print(f"Filesystem error during clone: {e}")
        if use_persistent:
            print("Falling back to ephemeral storage...")
            repo_dir = FALLBACK_REPO_DIR
            if not os.path.exists(repo_dir):
                subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)

    os.chdir(repo_dir)
    print("Changed to repo root:", os.getcwd())

    # Install only the dependencies needed for dataset generation
    # (no unsloth, no deepeval, no training deps)
    print("\nInstalling minimal dependencies...")

    # For OpenAI technique
    if TECHNIQUE == "openai":
        subprocess.run(
            ["pip", "install", "-q", "openai"],
            check=True,
        )
        print("  openai package installed.")

    # For Anthropic technique
    elif TECHNIQUE == "anthropic":
        subprocess.run(
            ["pip", "install", "-q", "anthropic"],
            check=True,
        )
        print("  anthropic package installed.")

    # For Ollama technique in Colab: warn the user
    if TECHNIQUE == "ollama":
        print()
        print("=" * 60)
        print("WARNING: Ollama is not available in Colab by default.")
        print("Use openai/anthropic techniques or run the Ollama server")
        print("separately and set OLLAMA_BASE_URL.")
        print("=" * 60)
        print()

    print("Setup complete.")

else:
    print("=" * 60)
    print("NOT RUNNING IN GOOGLE COLAB.")
    print("Open this notebook at https://colab.research.google.com/")
    print("with a GPU runtime for the full pipeline.")
    print("=" * 60)
    print("Continuing in local environment (limited support).")
    # Attempt to find local repo root
    curr = Path(os.getcwd()).resolve()
    repo_dir = None
    for parent in [curr] + list(curr.parents):
        if (parent / "ucore").exists() and (parent / "scripts").exists():
            repo_dir = parent
            break
    if repo_dir:
        print("Found local repo root:", repo_dir)
        os.chdir(str(repo_dir))
    else:
        print("Warning: Could not find local repo root.")

print("\nWorking directory:", os.getcwd())


In [ ]:
# @title 🚀 Run Dataset Generation
# @markdown Generate a training dataset using the configured technique.
# @markdown Output goes to `subjects/datasets/{NPC_KEY}/{TECHNIQUE}/train.jsonl`.

import os
import sys
import subprocess
import time
from pathlib import Path

python_bin = sys.executable
spec_path = f"subjects/{NPC_KEY}.json"
dataset_dir = f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}"
train_jsonl = f"{dataset_dir}/train.jsonl"

print("=" * 60)
print("DATASET GENERATION")
print(f"  NPC:       {NPC_KEY}")
print(f"  Technique: {TECHNIQUE}")
print(f"  Spec:      {spec_path}")
print(f"  Output:    {train_jsonl}")
print("=" * 60)

# --- Template ---
if TECHNIQUE == "template":
    cmd = [
        python_bin, "scripts/generate_dataset.py",
        spec_path,
        "--technique", "template",
        "--seed", str(TEMPLATE_SEED),
    ]
    print("\nRunning:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("=== GENERATION FAILED ===")
        print(result.stderr)
        raise SystemExit(1)
    print(result.stdout)

# --- Ollama ---
elif TECHNIQUE == "ollama":
    # Ensure Ollama server is running
    try:
        subprocess.run(["ollama", "--version"], check=True, capture_output=True)
        print("Ollama is installed.")
    except (FileNotFoundError, subprocess.CalledProcessError):
        print("Ollama not found. Installing...")
        subprocess.run(
            "curl -fsSL https://ollama.com/install.sh | sh",
            shell=True, check=True, timeout=120,
        )
        print("Ollama installed.")

    try:
        subprocess.run(["ollama", "list"], check=True, capture_output=True, timeout=10)
    except Exception:
        print("Starting Ollama server in background...")
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        time.sleep(5)

    # Pull model
    print(f"Pulling model: {OLLAMA_MODEL}...")
    subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True, timeout=300)

    # Generate
    cmd = [
        python_bin, "scripts/generate_dataset_ollama.py",
        spec_path,
        "--model", OLLAMA_MODEL,
        "--batch-size", "2",
        "--check-health",
    ]
    print("\nRunning:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("=== GENERATION FAILED ===")
        print(result.stderr)
        raise SystemExit(1)
    print(result.stdout)

# --- OpenAI ---
elif TECHNIQUE == "openai":
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        try:
            from google.colab import userdata  # type: ignore
            api_key = userdata.get("OPENAI_API_KEY")
        except Exception:
            pass

    if not api_key:
        print("ERROR: OPENAI_API_KEY not found.")
        print("Set it as a Colab secret (key icon in the left sidebar)")
        print("or as an environment variable, then re-run this cell.")
        raise SystemExit(1)

    os.environ["OPENAI_API_KEY"] = api_key
    cmd = [
        python_bin, "scripts/generate_dataset.py",
        spec_path,
        "--technique", "openai",
        "--model", OPENAI_MODEL,
    ]
    print("\nRunning:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("=== GENERATION FAILED ===")
        print(result.stderr)
        raise SystemExit(1)
    print(result.stdout)

# --- Anthropic ---
elif TECHNIQUE == "anthropic":
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        try:
            from google.colab import userdata  # type: ignore
            api_key = userdata.get("ANTHROPIC_API_KEY")
        except Exception:
            pass

    if not api_key:
        print("ERROR: ANTHROPIC_API_KEY not found.")
        print("Set it as a Colab secret (key icon in the left sidebar)")
        print("or as an environment variable, then re-run this cell.")
        raise SystemExit(1)

    os.environ["ANTHROPIC_API_KEY"] = api_key
    cmd = [
        python_bin, "scripts/generate_dataset.py",
        spec_path,
        "--technique", "anthropic",
        "--model", ANTHROPIC_MODEL,
    ]
    print("\nRunning:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("=== GENERATION FAILED ===")
        print(result.stderr)
        raise SystemExit(1)
    print(result.stdout)

else:
    print(f"Unknown technique: {TECHNIQUE}")
    print("Supported: template, ollama, openai, anthropic")
    raise SystemExit(1)

# Verify
if os.path.exists(train_jsonl):
    size_kb = os.path.getsize(train_jsonl) / 1024
    print(f"\nSUCCESS: Dataset generated at {train_jsonl} ({size_kb:.1f} KB)")
else:
    print(f"\nWARNING: Expected output not found at {train_jsonl}")
    print("Check the command output above for errors.")


In [ ]:
# @title 🔍 Preview Generated Dataset
# @markdown Show the first few lines of the generated dataset, total row count, and file size.

import os
import json

dataset_dir = f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}"
train_jsonl = f"{dataset_dir}/train.jsonl"
val_jsonl = f"{dataset_dir}/validation.jsonl"

print("=" * 60)
print("DATASET PREVIEW")
print("=" * 60)

# --- Training data ---
if os.path.exists(train_jsonl):
    size_bytes = os.path.getsize(train_jsonl)
    size_kb = size_bytes / 1024

    with open(train_jsonl, "r", encoding="utf-8") as f:
        lines = f.readlines()

    row_count = len(lines)
    print(f"\nFile: {train_jsonl}")
    print(f"Rows: {row_count}")
    print(f"Size: {size_kb:.1f} KB")

    print(f"\nFirst {min(3, row_count)} line(s):")
    print("-" * 40)
    for i, line in enumerate(lines[:3]):
        try:
            obj = json.loads(line.strip())
            preview = json.dumps(obj, indent=2)
            print(f"[Row {i + 1}]")
            print(preview[:600])
            if len(preview) > 600:
                print("... (truncated)")
            print()
        except json.JSONDecodeError:
            print(f"[Row {i + 1}] (invalid JSON)")
            print(line[:200])
            print()
else:
    print(f"\nTraining data not found at: {train_jsonl}")

# --- Validation data ---
if os.path.exists(val_jsonl):
    size_bytes = os.path.getsize(val_jsonl)
    size_kb = size_bytes / 1024

    with open(val_jsonl, "r", encoding="utf-8") as f:
        val_lines = f.readlines()

    print(f"\nValidation file: {val_jsonl}")
    print(f"Rows: {len(val_lines)}")
    print(f"Size: {size_kb:.1f} KB")
else:
    print(f"\nValidation data not found at: {val_jsonl}")


In [ ]:
# @title ⬇️ Download Generated Datasets
# @markdown Download train.jsonl and validation.jsonl to your local machine.

import os

print("=" * 60)
print("DOWNLOAD DATASETS")
print("=" * 60)

dataset_dir = f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}"

# Check runtime
is_colab = False
try:
    from google.colab import files  # type: ignore
    is_colab = True
except ImportError:
    pass

if not is_colab:
    print("Not running in Colab -- download via google.colab.files unavailable.")
    print("Datasets can be found at:")
    print(f"  {dataset_dir}/")
    raise SystemExit(0)

# Download train.jsonl
train_jsonl = os.path.join(dataset_dir, "train.jsonl")
if os.path.exists(train_jsonl):
    size_kb = os.path.getsize(train_jsonl) / 1024
    print(f"\nDownloading train.jsonl ({size_kb:.1f} KB)...")
    files.download(train_jsonl)
else:
    print(f"\ntrain.jsonl not found at {train_jsonl}.")

# Download validation.jsonl
val_jsonl = os.path.join(dataset_dir, "validation.jsonl")
if os.path.exists(val_jsonl):
    size_kb = os.path.getsize(val_jsonl) / 1024
    print(f"Downloading validation.jsonl ({size_kb:.1f} KB)...")
    files.download(val_jsonl)
else:
    print(f"validation.jsonl not found at {val_jsonl}.")

print("\nDownload complete.")
